In [0]:
%sql
USE CATALOG dbr_dev;

CREATE SCHEMA IF NOT EXISTS dbr_dev.parvinbadalov
COMMENT 'Parvin Badalov - Databricks Academy';

CREATE VOLUME IF NOT EXISTS dbr_dev.parvinbadalov.raw_files
COMMENT 'Raw file landing zone for academy labs';

CREATE VOLUME IF NOT EXISTS dbr_dev.parvinbadalov.checkpoints
COMMENT 'Checkpoint and schema tracking location for Auto Loader and streaming';

CREATE VOLUME IF NOT EXISTS dbr_dev.parvinbadalov.documents
COMMENT 'Document storage for RAG and AI capstone labs';

In [0]:
dbutils.fs.mkdirs("/Volumes/dbr_dev/parvinbadalov/raw_files/lab_01/")
dbutils.fs.mkdirs("/Volumes/dbr_dev/parvinbadalov/raw_files/lab_01/csv/")
dbutils.fs.mkdirs("/Volumes/dbr_dev/parvinbadalov/raw_files/lab_01/json/")

In [0]:
display(dbutils.fs.ls("/Volumes/dbr_dev/parvinbadalov/raw_files/lab_01/"))

In [0]:
catalog = "dbr_dev"
schema = "parvinbadalov"

raw_path = f"/Volumes/{catalog}/{schema}/raw_files/lab_01/csv/"

bronze_table = f"{catalog}.{schema}.lab01_fifa_players_bronze"
silver_table = f"{catalog}.{schema}.lab01_fifa_players_silver"
gold_table = f"{catalog}.{schema}.lab01_fifa_team_summary_gold"

print("Raw path:", raw_path)
print("Bronze table:", bronze_table)
print("Silver table:", silver_table)
print("Gold table:", gold_table)

In [0]:
dbutils.fs.mkdirs(raw_path)
display(dbutils.fs.ls(raw_path))

In [0]:
file_path = f"{raw_path}/fifa_world_cup_2026_player_performance.csv"

fifa_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

display(fifa_df)
fifa_df.printSchema()

In [0]:
raw_path = f"/Volumes/{catalog}/{schema}/raw_files/lab_01/csv/"

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, input_file_name, col

catalog = "dbr_dev"
schema = "parvinbadalov"

bronze_table = f"{catalog}.{schema}.lab01_fifa_players_bronze"

fifa_bronze_df = (
    fifa_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

fifa_bronze_df.write.mode("overwrite").format("delta").saveAsTable(bronze_table)

display(spark.table(bronze_table))

In [0]:
selected_df = fifa_bronze_df.select(
    "player_id",
    "player_name",
    "age",
    "nationality",
    "team",
    "position",
    "club_name",
    "market_value_eur",
    "match_id",
    "match_date",
    "opponent_team",
    "tournament_stage",
    "minutes_played",
    "goals",
    "assists",
    "shots",
    "shots_on_target",
    "expected_goals_xg",
    "expected_assists_xa",
    "pass_accuracy",
    "player_rating",
    "performance_score"
)

display(selected_df)

In [0]:
played_df = selected_df.filter("minutes_played > 0")

display(played_df)

In [0]:
attacking_impact_df = selected_df.filter(
    "goals > 0 OR assists > 0"
)

display(attacking_impact_df)

In [0]:
high_rating_df = selected_df.filter("player_rating >= 8.0")

display(high_rating_df)

In [0]:
team_region_data = [
    ("Argentina", "South America"),
    ("Brazil", "South America"),
    ("France", "Europe"),
    ("Germany", "Europe"),
    ("Spain", "Europe"),
    ("England", "Europe"),
    ("Portugal", "Europe"),
    ("Netherlands", "Europe"),
    ("Morocco", "Africa"),
    ("Senegal", "Africa"),
    ("Japan", "Asia"),
    ("South Korea", "Asia"),
    ("USA", "North America"),
    ("Mexico", "North America"),
    ("Canada", "North America"),
]

team_region_df = spark.createDataFrame(team_region_data, ["team", "region"])

display(team_region_df)

In [0]:
joined_df = (
    played_df
    .join(team_region_df, on="team", how="left")
)

display(joined_df)

# **Silver table**

In [0]:
from pyspark.sql.functions import col, round, when

silver_table = f"{catalog}.{schema}.lab01_fifa_players_silver"

fifa_silver_df = (
    fifa_bronze_df
    .dropDuplicates(["player_id", "match_id"])
    .fillna({
        "goals": 0,
        "assists": 0,
        "shots": 0,
        "shots_on_target": 0,
        "minutes_played": 0,
        "expected_goals_xg": 0.0,
        "expected_assists_xa": 0.0,
        "pass_accuracy": 0.0,
        "player_rating": 0.0,
        "performance_score": 0.0
    })
    .withColumn(
        "goal_contribution",
        col("goals") + col("assists")
    )
    .withColumn(
        "shots_accuracy_pct",
        when(col("shots") > 0, round((col("shots_on_target") / col("shots")) * 100, 2))
        .otherwise(0)
    )
    .withColumn(
        "minutes_category",
        when(col("minutes_played") >= 75, "High minutes")
        .when(col("minutes_played") >= 30, "Medium minutes")
        .otherwise("Low minutes")
    )
)

fifa_silver_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

display(spark.table(silver_table))

In [0]:
from pyspark.sql.functions import sum as spark_sum, avg, count, max

gold_table = f"{catalog}.{schema}.lab01_fifa_team_summary_gold"

team_summary_df = (
    fifa_silver_df
    .groupBy("team", "nationality")
    .agg(
        count("*").alias("player_match_records"),
        spark_sum("goals").alias("total_goals"),
        spark_sum("assists").alias("total_assists"),
        spark_sum("goal_contribution").alias("total_goal_contributions"),
        avg("player_rating").alias("avg_player_rating"),
        avg("performance_score").alias("avg_performance_score"),
        avg("pass_accuracy").alias("avg_pass_accuracy"),
        avg("shots_accuracy_pct").alias("avg_shots_accuracy_pct"),
        max("market_value_eur").alias("highest_market_value_eur")
    )
    .orderBy("total_goal_contributions", ascending=False)
)

team_summary_df.write.mode("overwrite").format("delta").saveAsTable(gold_table)

display(spark.table(gold_table))

In [0]:
%sql SHOW TABLES IN dbr_dev.parvinbadalov

In [0]:
import requests
import json
from pyspark.sql.functions import explode, arrays_zip, col

api_url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=52.23"
    "&longitude=21.01"
    "&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    "&timezone=Europe%2FWarsaw"
)

api_table = f"{catalog}.{schema}.lab01_api_weather_bronze"
api_raw_path = f"/Volumes/{catalog}/{schema}/raw_files/lab_01/api"

# Call API
response = requests.get(api_url)
response.raise_for_status()

weather_json = response.json()

# Save raw JSON to volume
dbutils.fs.mkdirs(api_raw_path)

dbutils.fs.put(
    f"{api_raw_path}/open_meteo_warsaw_weather.json",
    json.dumps(weather_json, indent=2),
    overwrite=True
)

# Convert nested JSON to Spark DataFrame
hourly = weather_json["hourly"]

weather_rows = list(
    zip(
        hourly["time"],
        hourly["temperature_2m"],
        hourly["relative_humidity_2m"],
        hourly["wind_speed_10m"]
    )
)

weather_df = spark.createDataFrame(
    weather_rows,
    ["time", "temperature_2m", "relative_humidity_2m", "wind_speed_10m"]
)

# Save as Delta table
weather_df.write.mode("overwrite").format("delta").saveAsTable(api_table)

display(weather_df)